# Reranker Evaluation: FAISS vs CrossEncoder

This notebook visualizes how the CrossEncoder reranker changes chunk ordering compared to raw FAISS scores.

Key questions:
- How much does the reranker change the ranking order?
- Does the top chunk after reranking match what we'd expect?
- Is the fine-tuned reranker better than the base model?

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.append("..")

from src.rag.pipeline import RAGPipeline
from src.rag.embedder import Embedder
from src.rag.indexer import FAISSIndexer
from sentence_transformers import CrossEncoder
import os

print("Loading pipeline...")
pipeline = RAGPipeline(data_dir="../data")
print("Pipeline ready.")

In [ ]:
# Load base model for comparison
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FINETUNED_MODEL_PATH = "../models/reranker"

base_reranker = CrossEncoder(BASE_MODEL, num_labels=1)

if os.path.exists(FINETUNED_MODEL_PATH):
    finetuned_reranker = CrossEncoder(FINETUNED_MODEL_PATH, num_labels=1)
    has_finetuned = True
    print("Fine-tuned reranker loaded.")
else:
    has_finetuned = False
    print("No fine-tuned model found — will compare FAISS vs base reranker only.")

In [ ]:
def get_ranking_comparison(question, top_k=10):
    """Get FAISS chunks and compare ranking with base and fine-tuned reranker."""
    # Get raw FAISS results
    chunks = pipeline.indexer.search(question, pipeline.embedder, top_k=top_k)

    pairs = [[question, c["text"]] for c in chunks]

    # Score with base reranker
    base_scores = base_reranker.predict(pairs)

    # Score with fine-tuned reranker if available
    finetuned_scores = finetuned_reranker.predict(pairs) if has_finetuned else None

    results = []
    for i, chunk in enumerate(chunks):
        results.append({
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"][:80] + "...",
            "faiss_score": chunk["score"],
            "base_reranker_score": float(base_scores[i]),
            "finetuned_reranker_score": float(finetuned_scores[i]) if finetuned_scores is not None else None,
        })
    return results


TEST_QUESTION = "What is virtual memory?"
comparison = get_ranking_comparison(TEST_QUESTION, top_k=10)
print(f"Retrieved {len(comparison)} chunks for: '{TEST_QUESTION}'")

In [ ]:
def rank_by(results, key):
    """Return chunk indices sorted by a given score key (descending)."""
    return [r["chunk_id"] for r in sorted(results, key=lambda x: x[key], reverse=True)]

faiss_ranking = rank_by(comparison, "faiss_score")
base_ranking = rank_by(comparison, "base_reranker_score")

print("FAISS ranking (top → bottom):")
for i, cid in enumerate(faiss_ranking):
    c = next(r for r in comparison if r["chunk_id"] == cid)
    print(f"  {i+1}. chunk_{cid:02d} | faiss={c['faiss_score']:.3f} | {c['text'][:60]}")

print("\nBase reranker ranking (top → bottom):")
for i, cid in enumerate(base_ranking):
    c = next(r for r in comparison if r["chunk_id"] == cid)
    print(f"  {i+1}. chunk_{cid:02d} | reranker={c['base_reranker_score']:.3f} | {c['text'][:60]}")

In [ ]:
# Visualize score comparison
n_cols = 3 if has_finetuned else 2
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 6))
fig.suptitle(f"Ranking Comparison\n'{TEST_QUESTION}'", fontsize=13, fontweight="bold")

chunk_labels = [f"chunk_{r['chunk_id']:02d}" for r in comparison]
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(comparison)))

def plot_ranking(ax, results, score_key, title, color):
    sorted_results = sorted(results, key=lambda x: x[score_key], reverse=True)
    labels = [f"chunk_{r['chunk_id']:02d}" for r in sorted_results]
    scores = [r[score_key] for r in sorted_results]
    bars = ax.barh(range(len(labels)), scores, color=color, alpha=0.85)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Score")
    ax.grid(axis="x", alpha=0.3)
    for bar, score in zip(bars, scores):
        ax.text(score + 0.001, bar.get_y() + bar.get_height()/2,
                f"{score:.3f}", va="center", fontsize=8)

plot_ranking(axes[0], comparison, "faiss_score", "FAISS Score", "#378ADD")
plot_ranking(axes[1], comparison, "base_reranker_score", "Base Reranker Score", "#00d2ff")

if has_finetuned:
    plot_ranking(axes[2], comparison, "finetuned_reranker_score", "Fine-tuned Reranker Score", "#00e5a0")

plt.tight_layout()
plt.savefig("reranker_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Rank shift analysis — how much did positions change?
faiss_positions = {r["chunk_id"]: i for i, r in enumerate(sorted(comparison, key=lambda x: x["faiss_score"], reverse=True))}
base_positions = {r["chunk_id"]: i for i, r in enumerate(sorted(comparison, key=lambda x: x["base_reranker_score"], reverse=True))}

print("Rank shift (FAISS → Base Reranker):")
print(f"{'Chunk':<12} {'FAISS Rank':<14} {'Reranker Rank':<16} {'Shift'}")
print("-" * 55)
for cid in faiss_positions:
    faiss_r = faiss_positions[cid] + 1
    base_r = base_positions[cid] + 1
    shift = faiss_r - base_r
    arrow = f"↑ {shift}" if shift > 0 else (f"↓ {abs(shift)}" if shift < 0 else "—")
    print(f"chunk_{cid:02d}     {faiss_r:<14} {base_r:<16} {arrow}")

In [ ]:
# Test multiple questions
QUESTIONS = [
    "What is virtual memory?",
    "How does process scheduling work?",
    "What is a deadlock?",
]

print("Top-1 chunk after FAISS vs Reranker:\n")
for q in QUESTIONS:
    results = get_ranking_comparison(q, top_k=10)
    faiss_top = sorted(results, key=lambda x: x["faiss_score"], reverse=True)[0]
    reranker_top = sorted(results, key=lambda x: x["base_reranker_score"], reverse=True)[0]
    changed = faiss_top["chunk_id"] != reranker_top["chunk_id"]
    print(f"Q: {q}")
    print(f"  FAISS top:    chunk_{faiss_top['chunk_id']:02d} | {faiss_top['text'][:70]}")
    print(f"  Reranker top: chunk_{reranker_top['chunk_id']:02d} | {reranker_top['text'][:70]}")
    print(f"  Ranking changed: {'YES ← reranker made a difference' if changed else 'no'}")
    print()